In [1]:
!wget https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2
!tar -xf LJSpeech-1.1.tar.bz2

--2026-05-04 23:17:42--  https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2
Resolving data.keithito.com (data.keithito.com)... 143.244.50.83, 2400:52e0:1a01::1108:1
Connecting to data.keithito.com (data.keithito.com)|143.244.50.83|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2748572632 (2.6G) [text/plain]
Saving to: ‘LJSpeech-1.1.tar.bz2’

LJSpeech-1.1.tar.bz 100%[===================>]   2.56G   164MB/s    in 17s     

2026-05-04 23:17:59 (154 MB/s) - ‘LJSpeech-1.1.tar.bz2’ saved [2748572632/2748572632]



In [3]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

metadata_path = "LJSpeech-1.1/metadata.csv"
wavs_path = "LJSpeech-1.1/wavs/"

df = pd.read_csv(metadata_path, sep='|', header=None, quoting=3)
df.columns = ["id", "transcription", "normalized_transcription"]
df = df[["id", "normalized_transcription"]].dropna()
df = df.head(2000)
df["file_name"] = wavs_path + df["id"] + ".wav"
df["normalized_transcription"] = df["normalized_transcription"].str.lower()

characters = set(char for val in df["normalized_transcription"] for char in val)
characters = sorted(list(characters))

char_to_num = keras.layers.StringLookup(vocabulary=characters, oov_token="")
num_to_char = keras.layers.StringLookup(vocabulary=char_to_num.get_vocabulary(), oov_token="", invert=True)

print(f"Словник має розмір: {char_to_num.vocabulary_size()} символів")

split = int(len(df) * 0.9)
train_df = df[:split]
val_df = df[split:]

frame_length = 256
frame_step = 160
fft_length = 384

def encode_single_sample(wav_file, label):
    file = tf.io.read_file(wav_file)
    audio, _ = tf.audio.decode_wav(file)
    audio = tf.squeeze(audio, axis=-1)
    audio = tf.cast(audio, tf.float32)
    spectrogram = tf.signal.stft(audio, frame_length=frame_length, frame_step=frame_step, fft_length=fft_length)
    spectrogram = tf.abs(spectrogram)
    spectrogram = tf.math.pow(spectrogram, 0.5)
    means = tf.math.reduce_mean(spectrogram, 1, keepdims=True)
    stddevs = tf.math.reduce_std(spectrogram, 1, keepdims=True)
    spectrogram = (spectrogram - means) / (stddevs + 1e-10)
    label = tf.strings.unicode_split(label, input_encoding="UTF-8")
    label = char_to_num(label)
    return spectrogram, label

batch_size = 16
train_dataset = tf.data.Dataset.from_tensor_slices((list(train_df["file_name"]), list(train_df["normalized_transcription"])))
train_dataset = train_dataset.map(encode_single_sample, num_parallel_calls=tf.data.AUTOTUNE).padded_batch(batch_size).prefetch(buffer_size=tf.data.AUTOTUNE)
val_dataset = tf.data.Dataset.from_tensor_slices((list(val_df["file_name"]), list(val_df["normalized_transcription"])))
val_dataset = val_dataset.map(encode_single_sample, num_parallel_calls=tf.data.AUTOTUNE).padded_batch(batch_size).prefetch(buffer_size=tf.data.AUTOTUNE)
def CTCLoss(y_true, y_pred):
    batch_len = tf.cast(tf.shape(y_true)[0], dtype="int64")
    input_length = tf.cast(tf.shape(y_pred)[1], dtype="int64")
    label_length = tf.cast(tf.shape(y_true)[1], dtype="int64")
    input_length = input_length * tf.ones(shape=(batch_len, 1), dtype="int64")
    label_length = label_length * tf.ones(shape=(batch_len, 1), dtype="int64")
    loss = keras.backend.ctc_batch_cost(y_true, y_pred, input_length, label_length)
    return loss

def build_model(input_dim, output_dim, rnn_layers=2, rnn_units=128):
    input_spectrogram = layers.Input((None, input_dim), name="input")
    x = layers.Reshape((-1, input_dim, 1))(input_spectrogram)
    x = layers.Conv2D(filters=32, kernel_size=[11, 41], strides=[2, 2], padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(filters=32, kernel_size=[11, 21], strides=[1, 2], padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Reshape((-1, x.shape[-2] * x.shape[-1]))(x)
    for _ in range(rnn_layers):
        x = layers.Bidirectional(layers.LSTM(rnn_units, return_sequences=True))(x)
        x = layers.Dropout(0.2)(x)

    x = layers.Dense(rnn_units * 2)(x)
    x = layers.ReLU()(x)
    x = layers.Dropout(0.2)(x)

    output = layers.Dense(output_dim + 1, activation="softmax", name="output")(x)
    model = keras.Model(inputs=input_spectrogram, outputs=output, name="DeepSpeech_2_LSTM")
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-4), loss=CTCLoss)
    return model
input_dim = fft_length // 2 + 1
output_dim = char_to_num.vocabulary_size()

model = build_model(input_dim, output_dim)
model.summary()
epochs = 10
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=epochs,
)
model_save_path = "/content/deepspeech2_model.h5"
model.save(model_save_path)
def decode_batch_predictions(pred):
    input_len = np.ones(pred.shape[0]) * pred.shape[1]
    results = keras.backend.ctc_decode(pred, input_length=input_len, greedy=True)[0][0]

    output_text = []
    for result in results:
        result = tf.strings.reduce_join(num_to_char(result)).numpy().decode("utf-8")
        output_text.append(result)
    return output_text

print("\nТестування на даних з датасету")
for batch in val_dataset.take(1):
    X, y = batch
    batch_predictions = model.predict(X)
    batch_predictions_text = decode_batch_predictions(batch_predictions)

    for i in range(3):
        print(f"Справжній текст: {tf.strings.reduce_join(num_to_char(y[i])).numpy().decode('utf-8')}")
        print(f"Передбачення: {batch_predictions_text[i]}\n")

def test_custom_audio(wav_path):
    print(f"\nТестування кастомного файлу: {wav_path}")
    try:
        spectrogram, _ = encode_single_sample(wav_path, "")
        spectrogram = tf.expand_dims(spectrogram, axis=0)
        prediction = model.predict(spectrogram)
        predicted_text = decode_batch_predictions(prediction)
        print(f"Розпізнаний текст: {predicted_text[0]}")
    except Exception as e:
        print(f"Помилка при обробці файлу: {e}")

test_custom_audio("/content/test_audio.wav")

Словник має розмір: 39 символів


Model: "DeepSpeech_2_LSTM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, None, 193)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_2 (Reshape)             │ (None, None, 193, 1)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, None, 97, 32)   │        14,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, None, 97, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_3 (ReLU)                  │ (None, None, 97, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, None, 49, 32)   │       236,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, None, 49, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_4 (ReLU)                  │ (None, None, 49, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_3 (Reshape)             │ (None, None, 1568)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, None, 256)      │     1,737,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, None, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (None, None, 256)      │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, None, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, None, 256)      │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_5 (ReLU)                  │ (None, None, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, None, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, None, 40)       │        10,280 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,459,272 (9.38 MB)

 Trainable params: 2,459,144 (9.38 MB)

 Non-trainable params: 128 (512.00 B)

Epoch 1/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 709s 6s/step - loss: 452.7720 - val_loss: 356.5678
Epoch 2/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 93s 819ms/step - loss: 316.4207 - val_loss: 377.7323
Epoch 3/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 93s 819ms/step - loss: 308.4733 - val_loss: 334.7180
Epoch 4/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 93s 824ms/step - loss: 299.7414 - val_loss: 320.2127
Epoch 5/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 92s 817ms/step - loss: 288.0199 - val_loss: 305.4134
Epoch 6/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 92s 817ms/step - loss: 274.3518 - val_loss: 283.2216
Epoch 7/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 93s 824ms/step - loss: 259.7907 - val_loss: 265.2118
Epoch 8/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 92s 815ms/step - loss: 245.0512 - val_loss: 248.3348
Epoch 9/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 92s 814ms/step - loss: 231.3383 - val_loss: 232.8083
Epoch 10/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 113s 1s/step - loss: 218.7009 - val_loss: 218.9178



Тестування на даних з датасету
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Справжній текст: as effectually to rebuke and abash the profane spirit of the more insolent and daring of the criminals.
Передбачення: s tlt n sftherinsnnths

Справжній текст: the lax discipline maintained in newgate was still further deteriorated by the presence of two other classes of prisoners who ought never to have been inmates of such a jail.
Передбачення: ssnntninsr thrtthrsnsohessrsonsrto ninss

Справжній текст: one of these were the criminal lunatics, who were at this time and for long previous continuously imprisoned there.
Передбачення: fthsrhrtr thistnrsonnslprson


Тестування кастомного файлу: /content/test_audio.wav
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 865ms/step
Розпізнаний текст: thenionhrssrnssrtthshrnnnrs
